<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Content_Based_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content Based Recommeder

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [3]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import linear_kernel

warnings.filterwarnings("ignore")

In [4]:
movie_data = pd.read_pickle(
    "/content/drive/MyDrive/Artifact/movie_data.pkl"
)

with open(
    "/content/drive/MyDrive/Artifact/tfidf_matrix.pkl",
    "rb"
) as f:
    tfidf_matrix = pickle.load(f)

with open(
    "/content/drive/MyDrive/Artifact/tfidf_vectorizer.pkl",
    "rb"
) as f:
    tfidf_vectorizer = pickle.load(f)

In [5]:
print("Movies:", movie_data.shape)
print("TF-IDF Shape:", tfidf_matrix.shape)

Movies: (87585, 10)
TF-IDF Shape: (87585, 10000)


In [6]:
title_to_index = (
    movie_data
    .reset_index(drop=True)
    .set_index("clean_title")
)

In [7]:
def find_movie(title):

    matches = movie_data[
        movie_data["clean_title"]
        .str.contains(
            title,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        print("Movie not found.")
        return None

    return matches[[
        "clean_title",
        "year",
        "genres"
    ]]

In [8]:
find_movie("batman")

,clean_title,year,genres
151,Batman Forever,1995.0,Action|Adventure|Comedy|Crime
584,Batman,1989.0,Action|Crime|Thriller
1341,Batman Returns,1992.0,Action|Crime
1506,Batman & Robin,1997.0,Action|Adventure|Fantasy|Thriller
3120,Batman: Mask of the Phantasm,1993.0,Animation|Children
8616,Batman,1966.0,Action|Adventure|Comedy
10004,Batman Begins,2005.0,Action|Crime|IMAX
17362,Batman,1943.0,Action|Adventure|Crime|Sci-Fi|Thriller
17363,Batman and Robin,1949.0,Action|Adventure|Crime|Drama|Sci-Fi
23479,The Batman vs. Dracula,2005.0,Action|Animation|Horror|Thriller


In [9]:
def recommend_content(title, top_n=10):

    matches = movie_data[
        movie_data["clean_title"]
        .str.lower() == title.lower()
    ]

    if matches.empty:
        print("Movie not found.")
        return None

    idx = matches.index[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx + 1],
        tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(cosine_scores)[::-1]

    similar_indices = similar_indices[1:top_n + 1]

    recommendations = movie_data.iloc[
        similar_indices
    ][[
        "clean_title",
        "genres",
        "year"
    ]].copy()

    recommendations["similarity_score"] = (
        cosine_scores[similar_indices]
    )

    return recommendations

In [10]:
recommend_content("Interstellar")

,clean_title,genres,year,similarity_score
37486,Space Cop,Action|Comedy|Sci-Fi,2016.0,0.499880
2220,2010: The Year We Make Contact,Sci-Fi,1984.0,0.497886
56683,Hyperlight,Sci-Fi,2018.0,0.475043
71994,Breach,Action|Sci-Fi,2020.0,0.466687
62748,The Search for Life in Space,Documentary,2016.0,0.455973
1527,Contact,Drama|Sci-Fi,1997.0,0.448968
20255,Gravity,Action|Sci-Fi|IMAX,2013.0,0.441677
72686,Heimat Is a Space in Time,Documentary,2019.0,0.432819
59170,The Wandering Earth,Sci-Fi,2019.0,0.432676
67464,Tenet,Action|Thriller,2020.0,0.428427


In [11]:
recommend_content("Toy Story")

,clean_title,genres,year,similarity_score
3021,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.892434
2264,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.806176
14815,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010.0,0.711669
39850,Finding Dory,Adventure|Animation|Comedy,2016.0,0.701200
4781,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001.0,0.700583
6259,Finding Nemo,Adventure|Animation|Children|Comedy,2003.0,0.696107
18314,Knick Knack,Animation|Children,1989.0,0.687346
8248,"Incredibles, The",Action|Adventure|Animation|Children|Comedy,2004.0,0.685074
19879,Monsters University,Adventure|Animation|Comedy,2013.0,0.661079
10812,Cars,Animation|Children|Comedy,2006.0,0.657805


In [12]:
recommend_content("The Matrix")

Movie not found.


In [13]:
def recommend_above_threshold(
    title,
    threshold=0.25
):

    matches = movie_data[
        movie_data["clean_title"]
        .str.lower() == title.lower()
    ]

    if matches.empty:
        return None

    idx = matches.index[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx+1],
        tfidf_matrix
    ).flatten()

    valid = np.where(
        cosine_scores >= threshold
    )[0]

    valid = valid[valid != idx]

    recommendations = (
        movie_data
        .iloc[valid]
        .copy()
    )

    recommendations["similarity_score"] = (
        cosine_scores[valid]
    )

    recommendations = (
        recommendations
        .sort_values(
            by="similarity_score",
            ascending=False
        )
    )

    return recommendations[[
        "clean_title",
        "genres",
        "year",
        "similarity_score"
    ]]

In [14]:
with open(
    "/content/drive/MyDrive/Artifact/title_to_index.pkl",
    "wb"
) as f:

    pickle.dump(title_to_index, f)